In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# ── CONFIG ────────────────────────────────────────
n_normal = 900      # normal log entries
n_anomaly = 100     # suspicious log entries

# ── NORMAL LOGS ───────────────────────────────────
normal_ips = [f"192.168.1.{i}" for i in range(1, 20)]
suspicious_ips = [f"45.33.{i}.{j}" for i in range(1,5) 
                  for j in range(1,5)]

actions = ["LOGIN_SUCCESS", "FILE_ACCESS", "DATA_READ",
           "LOGOUT", "PASSWORD_CHANGE"]

suspicious_actions = ["MULTIPLE_FAILED_LOGIN", "DATA_EXFILTRATION",
                      "PRIVILEGE_ESCALATION", "PORT_SCAN",
                      "BRUTE_FORCE_ATTEMPT"]

# Generate base timestamp
base_time = datetime(2024, 1, 1, 8, 0, 0)

logs = []

# Normal entries (business hours 8am-6pm)
for i in range(n_normal):
    timestamp = base_time + timedelta(
        days=random.randint(0, 30),
        hours=random.randint(0, 10),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    logs.append({
        'timestamp': timestamp,
        'ip_address': random.choice(normal_ips),
        'user_id': f"user_{random.randint(1, 50)}",
        'action': random.choice(actions),
        'bytes_transferred': random.randint(100, 5000),
        'response_time_ms': random.randint(50, 500),
        'failed_attempts': random.randint(0, 1),
        'hour': timestamp.hour,
        'is_anomaly': 0
    })

# Anomalous entries (odd hours, suspicious IPs)
for i in range(n_anomaly):
    timestamp = base_time + timedelta(
        days=random.randint(0, 30),
        hours=random.choice([0,1,2,3,22,23]),  # midnight hours
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    logs.append({
        'timestamp': timestamp,
        'ip_address': random.choice(suspicious_ips),
        'user_id': f"user_{random.randint(1, 50)}",
        'action': random.choice(suspicious_actions),
        'bytes_transferred': random.randint(50000, 500000),
        'response_time_ms': random.randint(1000, 5000),
        'failed_attempts': random.randint(3, 10),
        'hour': timestamp.hour,
        'is_anomaly': 1
    })

# Create DataFrame and shuffle
df = pd.DataFrame(logs)
df = df.sample(frac=1).reset_index(drop=True)
df = df.sort_values('timestamp').reset_index(drop=True)

# Save to CSV
df.to_csv(r'D:\Projects\log-anomaly-detector\network_logs.csv', 
          index=False)

print(f"✅ Generated {len(df)} log entries")
print(f"Normal: {n_normal} | Anomalies: {n_anomaly}")
print(df.head(10))

✅ Generated 1000 log entries
Normal: 900 | Anomalies: 100
            timestamp    ip_address  user_id                action  \
0 2024-01-01 08:34:35  192.168.1.14   user_1         LOGIN_SUCCESS   
1 2024-01-01 08:37:50  192.168.1.16  user_45       PASSWORD_CHANGE   
2 2024-01-01 08:42:22  192.168.1.11  user_31                LOGOUT   
3 2024-01-01 09:12:57  192.168.1.15   user_9         LOGIN_SUCCESS   
4 2024-01-01 09:14:08   192.168.1.5  user_45           FILE_ACCESS   
5 2024-01-01 09:47:03   192.168.1.3  user_32         LOGIN_SUCCESS   
6 2024-01-01 10:16:15     45.33.4.3  user_16  PRIVILEGE_ESCALATION   
7 2024-01-01 10:32:39     45.33.4.1  user_20             PORT_SCAN   
8 2024-01-01 11:10:40  192.168.1.10   user_6                LOGOUT   
9 2024-01-01 11:14:46   192.168.1.5  user_14         LOGIN_SUCCESS   

   bytes_transferred  response_time_ms  failed_attempts  hour  is_anomaly  
0               4436               419                1     8           0  
1               136